# Spiking MNIST with NengoDL

This notebook demonstrates how to train a spiking convolutional neural network (CNN) to classify MNIST handwritten digits using the **nengo-dl** PyTorch library.

The key idea is a two-phase approach:
- **Training**: run in *rate mode* (1 timestep), where LIF neurons act as smooth rate-coded activations. This makes training easy with standard backpropagation.
- **Inference**: run in *spiking mode* (30 timesteps), where LIF neurons fire discrete spikes. Spike trains are accumulated over time through a low-pass filter to produce final class scores.

This technique lets us get the biological realism of spiking neurons while still using gradient-based training.

In [1]:
%matplotlib inline
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import nengo
import nengo_dl

In [ ]:
# Load MNIST dataset
def _prepare_images(images):
    images = np.asarray(images, dtype=np.float32)
    images = images.reshape(images.shape[0], -1)
    return images


def load_mnist():
    """Load MNIST from a stable local cache, falling back to torchvision."""
    for path in [Path.home() / ".keras" / "datasets" / "mnist.npz", Path("/tmp/mnist_data.npz")]:
        if path.exists():
            with np.load(path) as data:
                train_images = _prepare_images(data["x_train"])
                train_labels = np.asarray(data["y_train"], dtype=np.int64)
                test_images = _prepare_images(data["x_test"])
                test_labels = np.asarray(data["y_test"], dtype=np.int64)
            return train_images, train_labels, test_images, test_labels, str(path)

    # Last resort: download into a project-local cache. This requires internet
    # the first time, but then works offline from ./data/MNIST.
    from torchvision.datasets import MNIST

    data_root = Path("data")
    train = MNIST(root=data_root, train=True, download=True)
    test = MNIST(root=data_root, train=False, download=True)

    train_images = _prepare_images(train.data.numpy())
    train_labels = train.targets.numpy().astype(np.int64)
    test_images = _prepare_images(test.data.numpy())
    test_labels = test.targets.numpy().astype(np.int64)
    return train_images, train_labels, test_images, test_labels, str(data_root / "MNIST")


train_images, train_labels, test_images, test_labels, mnist_source = load_mnist()

print(f"Loaded MNIST from: {mnist_source}")
print(f"Train: {train_images.shape}, labels: {train_labels.shape}")
print(f"Test:  {test_images.shape},  labels: {test_labels.shape}")

# Visualize a few examples
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(train_images[i].reshape(28, 28), cmap="gray")
    ax.set_title(f"Label: {train_labels[i]}")
    ax.axis("off")
plt.suptitle("Sample MNIST images")
plt.tight_layout()
plt.show()

## Building the Spiking CNN

We construct a convolutional network with the following architecture:

```
Input (784)
  → Conv2d(1, 32, 3)       # (28,28,1) → (26,26,32)
  → LIF neurons
  → Conv2d(32, 64, 3, s=2) # (26,26,32) → (12,12,64)
  → LIF neurons
  → Conv2d(64, 128, 3, s=2)# (12,12,64) → (5,5,128)
  → LIF neurons
  → Linear(3200, 10)       # class scores
```

Key design choices:
- **`amplitude=0.01`** on LIF neurons keeps outputs in a reasonable range during rate-mode training.
- **`stateful=False`** resets neuron state between samples, which is important for classification tasks.
- Two probes are attached to the output: `out_p` (unfiltered, for training) and `out_p_filt` (low-pass filtered with `synapse=0.1`, for spiking inference).

In [3]:
with nengo.Network(seed=0) as net:
    # Configure default parameters for cleaner training
    net.config[nengo.Ensemble].max_rates = nengo.dists.Choice([100])
    net.config[nengo.Ensemble].intercepts = nengo.dists.Choice([0])
    net.config[nengo.Connection].synapse = None

    # LIF neurons with amplitude=0.01 (keeps outputs in [0, 1] range)
    neuron_type = nengo.LIF(amplitude=0.01)

    # stateful=False: reset state between samples (important for classification)
    # lif_smoothing=0.005: use smooth SoftLIF rate approximation during training
    # (gives non-zero gradients everywhere, avoiding the dead-neuron problem)
    nengo_dl.configure_settings(stateful=False, lif_smoothing=0.005)

    # Input node
    inp = nengo.Node(np.zeros(28 * 28))

    # Conv layer 1: (28, 28, 1) → (26, 26, 32)
    x = nengo_dl.Layer(nn.Conv2d(1, 32, 3))(inp, shape_in=(28, 28, 1))
    x = nengo_dl.Layer(neuron_type)(x)

    # Conv layer 2: (26, 26, 32) → (12, 12, 64)  [stride=2]
    x = nengo_dl.Layer(nn.Conv2d(32, 64, 3, stride=2))(x, shape_in=(26, 26, 32))
    x = nengo_dl.Layer(neuron_type)(x)

    # Conv layer 3: (12, 12, 64) → (5, 5, 128)  [stride=2]
    x = nengo_dl.Layer(nn.Conv2d(64, 128, 3, stride=2))(x, shape_in=(12, 12, 64))
    x = nengo_dl.Layer(neuron_type)(x)

    # Linear readout: 3200 → 10 class scores
    out = nengo_dl.Layer(nn.Linear(x.size_out, 10))(x)

    # Two probes:
    # out_p: unfiltered (used during rate-mode training)
    out_p = nengo.Probe(out, label="out_p")
    # out_p_filt: low-pass filtered (used during spiking inference)
    out_p_filt = nengo.Probe(out, synapse=0.1, label="out_p_filt")

print(f"Network built successfully")
print(f"  Conv1 output size: {26*26*32}")
print(f"  Conv2 output size: {12*12*64}")
print(f"  Conv3 output size: {5*5*128}")
print(f"  Linear input size: {x.size_out}")

Network built successfully
  Conv1 output size: 21632
  Conv2 output size: 9216
  Conv3 output size: 3200
  Linear input size: 3200


## Preparing the Data

We need two different data formats:

1. **Training data (1 timestep)**: Each image is presented for a single timestep. Shape: `(n_samples, 1, 784)`. Labels shape: `(n_samples, 1, 1)`. This is rate-mode training — fast and differentiable.

2. **Test data (30 timesteps)**: Each image is repeated for 30 timesteps. Shape: `(n_samples, 30, 784)`. This allows spiking neurons to fire over multiple steps, accumulating evidence before making a classification decision.

In [4]:
minibatch_size = 200

# Training data: single timestep (rate-mode approximation)
# Shape: (n_samples, 1, 784)
train_images_1step = train_images[:, None, :]       # (60000, 1, 784)
train_labels_1step = train_labels[:, None, None]    # (60000, 1, 1)

# Test data: 30 timesteps (spiking inference - accumulate spikes)
n_steps = 30
test_images_nstep = np.tile(test_images[:, None, :], (1, n_steps, 1))    # (10000, 30, 784)
test_labels_nstep = np.tile(test_labels[:, None, None], (1, n_steps, 1)) # (10000, 30, 1)

print("Training data (1 timestep):")
print(f"  images: {train_images_1step.shape}")
print(f"  labels: {train_labels_1step.shape}")
print()
print(f"Test data ({n_steps} timesteps for spiking inference):")
print(f"  images: {test_images_nstep.shape}")
print(f"  labels: {test_labels_nstep.shape}")

Training data (1 timestep):
  images: (60000, 1, 784)
  labels: (60000, 1, 1)

Test data (30 timesteps for spiking inference):
  images: (10000, 30, 784)
  labels: (10000, 30, 1)


## Evaluating Before Training

We first check the network's accuracy with random (untrained) weights. We expect roughly 10% accuracy (chance level for 10 classes).

In [5]:
# Classification accuracy function
# y_true: (batch, n_steps, 1) int labels
# y_pred: (batch, n_steps, 10) float logits
def classification_accuracy(y_true, y_pred):
    """Compute classification accuracy on the last timestep."""
    # y_true: (batch, n_steps, 1), y_pred: (batch, n_steps, 10)
    true_last = y_true[:, -1, 0].astype(np.int64)
    pred_last = np.argmax(y_pred[:, -1, :], axis=-1)
    return np.mean(true_last == pred_last)

# Evaluate on a subset before training
n_eval = minibatch_size  # evaluate on one minibatch
with nengo_dl.Simulator(net, minibatch_size=minibatch_size, seed=0) as sim:
    eval_images = test_images_nstep[:n_eval]
    eval_labels = test_labels_nstep[:n_eval]

    sim.run_steps(n_steps, data={inp: eval_images}, inference_mode="spiking")
    preds = sim.data[out_p_filt]  # (n_eval, n_steps, 10)

    acc = classification_accuracy(eval_labels, preds)
    print(f"Accuracy before training: {acc:.4f}  (chance = 0.10)")

Accuracy before training: 0.0800  (chance = 0.10)


## Training the Network

We train using **rate mode** (1 timestep per sample), which treats LIF neurons as smooth rate-coded activations. This is crucial because:
- Spiking neurons are not directly differentiable (their output is a binary spike train).
- In rate mode, LIF neurons approximate their average firing rate, giving smooth gradients.
- After training, the learned weights transfer to spiking mode.

We use **sparse categorical cross-entropy** as the loss function, which works directly with integer class labels. The optimizer is RMSprop, matching the online NengoDL example.

Training for 10 epochs over all 60,000 MNIST training images typically achieves ~98% rate-mode accuracy.

In [ ]:
# Sparse categorical crossentropy for one-hot classification
# Note: _compute_loss calls loss_fn(pred, target) - PyTorch convention (pred first)
def sparse_crossentropy(y_pred, y_true):
    """Sparse cross-entropy: y_true has integer class labels."""
    import torch.nn.functional as F
    # y_pred: (batch, n_steps, 10) logits
    # y_true: (batch, n_steps, 1) int labels as float
    batch, n_steps_local, n_classes = y_pred.shape
    labels = y_true[:, :, 0].long()    # (batch, n_steps)
    logits = y_pred.reshape(-1, n_classes)
    targets = labels.reshape(-1)
    return F.cross_entropy(logits, targets)

do_training = True  # Set to False to skip training
params_path = "./mnist_params_pt_online_scale"

if do_training:
    with nengo_dl.Simulator(net, minibatch_size=minibatch_size, seed=0) as sim:
        n_params = sum(p.numel() for p in sim.trainable_params())
        print(f"Total trainable parameters: {n_params:,}")

        sim.compile(
            optimizer='rmsprop',
            loss={out_p: sparse_crossentropy},
        )

        # Train for 10 epochs on all 60,000 training images
        # (each image is a single timestep in rate mode)
        history = sim.fit(
            {inp: train_images_1step},
            {out_p: train_labels_1step.astype(np.float32)},
            n_steps=1,
            epochs=10,
        )

        # Save trained parameters
        sim.save_params(params_path)
        print(f"Parameters saved to {params_path}.npz")

    # Plot training loss
    plt.figure(figsize=(8, 4))
    plt.plot(history['loss'])
    plt.title("Training loss (cross-entropy)")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.tight_layout()
    plt.show()
else:
    print("Skipping training (do_training=False)")
    print("To train, set do_training=True")

## Evaluating After Training (Rate Mode)

We first evaluate accuracy in rate mode (1 timestep), which matches the training conditions. This gives us an upper bound on performance — spiking mode accuracy is typically slightly lower due to the discrete nature of spikes.

In [7]:
# Evaluate accuracy in rate mode (1 timestep) on test set
with nengo_dl.Simulator(net, minibatch_size=minibatch_size, seed=0) as sim:
    if do_training:
        sim.load_params(params_path)

    # Evaluate on first 1000 test images (5 batches of 200)
    n_test_eval = 1000
    correct = 0
    total = 0

    for start in range(0, n_test_eval, minibatch_size):
        end = start + minibatch_size
        if end > n_test_eval:
            break
        batch_imgs = test_images[start:end, None, :]      # (200, 1, 784)
        batch_labs = test_labels[start:end]

        sim.run_steps(1, data={inp: batch_imgs}, inference_mode="rate")
        preds_batch = sim.data[out_p]  # (200, 1, 10)
        pred_labels = np.argmax(preds_batch[:, 0, :], axis=1)
        correct += np.sum(pred_labels == batch_labs)
        total += len(batch_labs)

    rate_acc = correct / total
    print(f"Rate-mode accuracy on {total} test images: {rate_acc:.4f}")

Rate-mode accuracy on 1000 test images: 0.9850


## Evaluating After Training (Spiking Mode)

Now we evaluate in **spiking mode** using 30 timesteps. In this mode:
- LIF neurons fire discrete all-or-nothing spikes at each timestep.
- The filtered probe (`out_p_filt`, `synapse=0.1`) applies an exponential low-pass filter that smooths the spike trains, accumulating evidence over time.
- We use the final filtered class scores to make a prediction, matching the online NengoDL example.

More timesteps generally yields higher accuracy (more spikes to accumulate), but 30 is a good tradeoff between accuracy and compute cost.

In [8]:
# Evaluate accuracy in spiking mode (n_steps=30) on test set
with nengo_dl.Simulator(net, minibatch_size=minibatch_size, seed=0) as sim:
    if do_training:
        sim.load_params(params_path)

    n_test_eval = 1000
    correct = 0
    total = 0

    for start in range(0, n_test_eval, minibatch_size):
        end = start + minibatch_size
        if end > n_test_eval:
            break
        batch_imgs = test_images_nstep[start:end]   # (200, 30, 784)
        batch_labs = test_labels[start:end]

        sim.run_steps(n_steps, data={inp: batch_imgs}, inference_mode="spiking")
        # Use filtered probe (out_p_filt) for spiking inference
        preds_batch = sim.data[out_p_filt]  # (200, 30, 10)
        # Use the final filtered timestep for prediction. The filter has already
        # accumulated evidence over time, so averaging would over-weight the
        # initial transient.
        pred_labels = np.argmax(preds_batch[:, -1, :], axis=1)
        correct += np.sum(pred_labels == batch_labs)
        total += len(batch_labs)

    spiking_acc = correct / total
    print(f"Spiking-mode accuracy on {total} test images: {spiking_acc:.4f}")
    print(f"(Rate-mode was {rate_acc:.4f})")

Spiking-mode accuracy on 1000 test images: 0.9840
(Rate-mode was 0.9850)


## Visualizing Predictions Over Time

One of the unique properties of spiking networks is that predictions evolve over time as spikes accumulate. Here we plot class probabilities at each of the 30 timesteps for a few test images.

You should see the correct class (highlighted) emerge from the noise and dominate as more timesteps are processed.

In [ ]:
# Visualize network predictions over time for a few test images
with nengo_dl.Simulator(net, minibatch_size=minibatch_size, seed=0) as sim:
    if do_training:
        sim.load_params(params_path)

    # Run on first minibatch of test images (30 timesteps)
    sim.run_steps(n_steps, data={inp: test_images_nstep[:minibatch_size]}, inference_mode="spiking")
    preds_time = sim.data[out_p_filt]  # (200, 30, 10)

# Show 5 example predictions
fig, axes = plt.subplots(5, 2, figsize=(12, 15))
for i in range(5):
    # Left: input image
    axes[i, 0].imshow(test_images[i].reshape(28, 28), cmap="gray")
    axes[i, 0].axis("off")
    true_label = test_labels[i]
    pred_label = np.argmax(preds_time[i, -1, :])
    axes[i, 0].set_title(f"True: {true_label}, Pred: {pred_label}")

    # Right: class scores over time
    # Apply softmax to get probabilities
    scores = preds_time[i]  # (30, 10)
    scores_exp = np.exp(scores - scores.max(axis=1, keepdims=True))
    probs = scores_exp / scores_exp.sum(axis=1, keepdims=True)

    for cls in range(10):
        alpha = 1.0 if cls == true_label else 0.3
        lw = 2 if cls == true_label else 0.8
        axes[i, 1].plot(probs[:, cls], label=str(cls), alpha=alpha, lw=lw)

    axes[i, 1].set_xlabel("Timestep")
    axes[i, 1].set_ylabel("Probability")
    axes[i, 1].set_title(f"Class probabilities over {n_steps} timesteps")
    axes[i, 1].legend(loc="upper left", fontsize=7, ncol=2)

plt.suptitle("Spiking network predictions on MNIST", fontsize=14)
plt.tight_layout()
plt.show()

# Plot aggregate accuracy and margin over time on a larger subset
n_curve_eval = 1000
curve_correct = np.zeros(n_steps, dtype=np.int64)
curve_total = 0
curve_margin_sum = np.zeros(n_steps, dtype=np.float64)

with nengo_dl.Simulator(net, minibatch_size=minibatch_size, seed=0) as sim:
    if do_training:
        sim.load_params(params_path)

    for start in range(0, n_curve_eval, minibatch_size):
        end = start + minibatch_size
        if end > n_curve_eval:
            break
        batch_imgs = test_images_nstep[start:end]
        batch_labs = test_labels[start:end]

        sim.run_steps(n_steps, data={inp: batch_imgs}, inference_mode="spiking")
        preds_batch = sim.data[out_p_filt]
        pred_labels = np.argmax(preds_batch, axis=2)
        curve_correct += np.sum(pred_labels == batch_labs[:, None], axis=0)

        sorted_logits = np.sort(preds_batch, axis=2)
        margins = sorted_logits[:, :, -1] - sorted_logits[:, :, -2]
        curve_margin_sum += margins.sum(axis=0)
        curve_total += len(batch_labs)

accuracy_by_t = curve_correct / curve_total
margin_by_t = curve_margin_sum / curve_total

fig, ax1 = plt.subplots(figsize=(8, 4))
time = np.arange(1, n_steps + 1)
ax1.plot(time, accuracy_by_t, color="tab:blue", label="accuracy")
ax1.set_xlabel("Timestep")
ax1.set_ylabel("Accuracy", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.set_ylim(0, 1)

ax2 = ax1.twinx()
ax2.plot(time, margin_by_t, color="tab:orange", label="top-2 logit margin")
ax2.set_ylabel("Top-2 logit margin", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

plt.title(f"Spiking accuracy over {n_steps} timesteps")
fig.tight_layout()
plt.show()

## Summary

This notebook demonstrated the full pipeline for training a spiking CNN on MNIST with NengoDL:

1. **Build** a convolutional network with LIF spiking neurons using `nengo_dl.Layer`.
2. **Train** in rate mode (1 timestep) using standard backpropagation with sparse cross-entropy loss.
3. **Evaluate** in spiking mode (30 timesteps) by accumulating spikes through a low-pass filter.

The rate-mode training trick allows efficient gradient-based optimization while the resulting network runs as a genuine spiking neural network at inference time — making it suitable for neuromorphic hardware deployment.

In [10]:
print("=== MNIST Spiking Network Summary ===")
print()
print("Architecture:")
print("  Input (784) → Conv2d(1,32,3) → LIF → Conv2d(32,64,3,s=2) → LIF")
print("              → Conv2d(64,128,3,s=2) → LIF → Linear(3200,10) → Output")
print()
print("Training approach:")
print("  - Rate-mode (n_steps=1): differentiable, LIF treated as rate neurons")
print("  - Loss: sparse categorical cross-entropy")
print("  - Optimizer: RMSprop, 10 epochs, minibatch_size=200")
print()
print("Inference:")
print("  - Spiking-mode (n_steps=30): discrete spike trains accumulated over time")
print("  - Low-pass filter (tau=0.1s) smooths spike trains for final classification")
print()
if do_training:
    print(f"Results:")
    print(f"  Rate-mode accuracy:    {rate_acc:.4f}")
    print(f"  Spiking-mode accuracy: {spiking_acc:.4f}")

=== MNIST Spiking Network Summary ===

Architecture:
  Input (784) → Conv2d(1,32,3) → LIF → Conv2d(32,64,3,s=2) → LIF
              → Conv2d(64,128,3,s=2) → LIF → Linear(3200,10) → Output

Training approach:
  - Rate-mode (n_steps=1): differentiable, LIF treated as rate neurons
  - Loss: sparse categorical cross-entropy
  - Optimizer: RMSprop, 10 epochs, minibatch_size=200

Inference:
  - Spiking-mode (n_steps=30): discrete spike trains accumulated over time
  - Low-pass filter (tau=0.1s) smooths spike trains for final classification

Results:
  Rate-mode accuracy:    0.9850
  Spiking-mode accuracy: 0.9840
